In [ ]:
from itertools import product
import pandas as pd
import cobra
import json

Dictionary of all M-metabolites used in expression module

In [91]:
nucleotides = ['a', 'c', 'g', 'u']
nps = ['mp', 'tp', 'dp']
atop_hydrolysis = ['atp', 'adp', 'h', 'pi', 'h2o']

aa_s = ['ala_L', 'arg_L', 'asn_L', 'asp_L', 'cys_L', 'gln_L', 'glu_L', 'gly', 'his_L','ile_L','leu_L','lys_L','phe_L',
        'pro_L','ser_L', 'thr_L','trp_L','tyr_L','val_L', 'met_L']


required_metabolites = {'c': [], 'l': [], 'm': [], 'r': [], 'e': [], 'x': [], 'n': [], 'g': [], 'i': [], 'pm': []}

required_metabolites['c'] += ['ppi[c]', 'amet[c]', 'ahcys[c]', 'g6p[c]', 'chsterol[c]', 
                             'clpn_hs[c]', 'pail_hs[c]', 'pchol_hs[c]','pe_hs[c]','pglyc_hs[c]','ps_hs[c]',
                              'sphmyln_hs[c]']

required_metabolites['n'] += ['ppi[n]', 'amet[n]', 'ahcys[n]', 'adp[n]', 'gdp[n]']

required_metabolites['r'] += ['h2o2[r]', 'hdca[r]', 'gpi_hs[r]', 'o2[r]', 'udpacgal[r]', 
                              'udpgal[r]', 'uacgam[r]', 'udp[r]'] 
                              #'gpi_sig[r]', 'm_em_3gacpail_hs[r]', 'm_em_3gacpail_prot_hs[r]', 'pre_prot[r]']
required_metabolites['g'] += ['udpacgal[g]', 'udpgal[g]', 'uacgam[g]', 'h[g]', 'udp[g]']
required_metabolites['l'] += ['o2[l]', 'h2o2[l]', 'udpacgal[l]', 'udp[l]']


# add nucleotides
required_metabolites['c'] += [''.join(k) + '[c]' for k in list(product(nucleotides, nps))]
required_metabolites['n'] += [n + 'tp[n]' for n in nucleotides] + [n + 'mp[n]' for n in nucleotides] + ['cdp[n]']
required_metabolites['n'] += ['d' + n + 'tp[n]' for n in nucleotides][:-1] + ['dttp[n]']
# add amino acids
for comp in ['c', 'm', 'l', 'x', 'n', 'r']:
    required_metabolites[comp] += [a + '[' + comp + ']' for a in aa_s]

# add atp hydrolysis metabolites
for comp in ['m', 'x', 'r', 'l', 'c', 'n']:
    required_metabolites[comp] += [hydro + '[' + comp + ']' for hydro in atop_hydrolysis]

required_metabolites = {k: sorted(set(v)) for k, v in required_metabolites.items()}

In [92]:
json.dump(required_metabolites, open(build_files_path + "required_metabolic_model_metabolites.json", 'w' ))

Generate dataframe of relevant information for metabolites

In [30]:
model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')

In [93]:
def bool_metabolite(m_id, compartment):
    try: 
        m = model.metabolites.get_by_id(m_id + '[' + compartment + ']')
#         print(m_id + '[' + compartment + ']')
        return True, m 
    except:
        return False, None

In [94]:
metab_ids = [item for sublist in list(required_metabolites.values()) for item in sublist]
metab_ids = sorted(set([i.split('[')[0] for i in metab_ids]))

counter = 0
compartments = sorted(required_metabolites.keys())
rmd = pd.DataFrame(columns = ['id', 'name', 'charge', 'elements', 'formula'])

for m_id in metab_ids:
    m_found = False
    counter_ = 0
    while not m_found: # find the first compartment the metabolite is in and get the information
#         print(counter_)
        compartment = compartments[counter_]
        m_found, m = bool_metabolite(m_id, compartment)
        counter_ += 1
        if counter_ > len(compartments):
            raise ValueError('metabolite note found')
    rmd.loc[counter,:] = [m_id, m.name, m.charge, m.elements, m.formula]
    counter += 1

rmd.index = rmd.id
rmd.drop(columns = ['id'], inplace = True)
# rmd.to_csv(build_files_path + 'required_metabolic_model_metabolites.csv')

In [95]:
build_files_path = '/data2/hratch/human_me/build_files/'
rmd.to_csv(build_files_path + 'required_metabolic_model_metabolites.csv')


In [78]:
# 1) make dict + dataframe (compartment independent)
# 2) add cytoplasmic metabolites first -- if not there, first try to transport from another compartment, else add exchange + transport
# 3) then do remaining compartments